In [ ]:
######################################
#######  Phase 4. Two-way LMM  #######
######################################

import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import os
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning

warnings.simplefilter('ignore', ConvergenceWarning)

base_dir = './'
raw_data_path = './HarvestData.xlsx'
phase2_path = os.path.join(base_dir, 'Phase2_Heritability_LMM.csv')

print("Loading data..")
df_raw = pd.read_excel(raw_data_path)
df_h2 = pd.read_csv(phase2_path)

# Run only passed traits through phase 2
passed_traits = df_h2[df_h2['Selection'] == 'Pass']['Trait'].tolist()
phase4_results = []
print(f"Analyzing {len(passed_traits)} traits for treatment and GxE effects")

# LMM Fitting and extract variance components
for trait in passed_traits:
    df_clean = df_raw[['Genotype', 'Treatment', trait]].dropna().copy()
    df_clean.columns = ['Genotype', 'Treatment', 'Value']
    
    # Set the reference level in Trt1
    df_clean['Treatment'] = pd.Categorical(df_clean['Treatment'], categories=['Trt1', 'Trt2'])
    
    try:
        #################################################
        # [LMM Modeling structure]
        # - Fixed effect: Treatment
        # - Random effect: Genotype (Intercept)
        # - Random slope: Treatment (GxE Interaction)
        #################################################

        model = smf.mixedlm("Value ~ C(Treatment)", 
                            data=df_clean, 
                            groups=df_clean["Genotype"], 
                            re_formula="~C(Treatment)")
        result = model.fit(method='bfgs', maxiter=2000)
         
        p_treatment = result.pvalues.get("C(Treatment)[T.Trt2]", np.nan) # Fixed Effect: Does Trt2 treatment give significant change in average
        cov_matrix = result.cov_re  # Variance components
        var_G = max(cov_matrix.iloc[0, 0], 0)   
        var_GxE = max(cov_matrix.iloc[1, 1], 0) if cov_matrix.shape[0] > 1 else 0   
        var_Error = max(result.scale, 0)
        
        # calculate variance component proportions
        total_var = var_G + var_GxE + var_Error
        
        if total_var > 0:
            vcp_G = (var_G / total_var) * 100
            vcp_GxE = (var_GxE / total_var) * 100
            vcp_Error = (var_Error / total_var) * 100
        else:
            vcp_G, vcp_GxE, vcp_Error = 0, 0, 0
            
        phase4_results.append({
            'Trait': trait,
            'Treatment_p_value': p_treatment,
            'Var_Genotype(%)': round(vcp_G, 2),
            'Var_GxE(%)': round(vcp_GxE, 2),
            'Var_Error(%)': round(vcp_Error, 2),
            'Total_Variance': round(total_var, 4)
        })

    except Exception as e:
        print(f"Convergence/Fitting error for {trait}: {e}")
        continue

# Save the file
df_phase4 = pd.DataFrame(phase4_results)

# Order in high GxE_var
df_phase4 = df_phase4.sort_values(by='Var_GxE(%)', ascending=False)

save_path_phase4 = os.path.join(base_dir, 'Phase4_LMM_Effect_Structure.csv')
df_phase4.to_csv(save_path_phase4, index=False, encoding='utf-8-sig')

print("-" * 50)
print("Phase 4 LMM Analysis complete.")
print(f"Results saved to: {save_path_phase4}")
print("-" * 50)
print("\n[Top 5 Traits by GxE Interaction Variance Proportion]")
print(df_phase4[['Trait', 'Treatment_p_value', 'Var_Genotype(%)', 'Var_GxE(%)']].head())